In [1]:
# ============================================================
# 步骤1：导入并检查GPU（在Colab上运行时取消下一行安装的注释）
# ============================================================
# !pip install torch transformers scikit-learn pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import re

# 检查设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}") # 如果能正常打印，说明配置成功


Using device: cuda


In [2]:
# ============================================================
# 步骤2：准备数据（使用之前清理和加载好的数据）
# ============================================================
# 先加载数据集得到 unique_texts 和 unique_labels

import json
import re
import os
import glob
import jieba
from collections import Counter

print("="*60)
print("完整数据集实验 (谣言386 vs 事实81)")
print("="*60)

# ========== 1. 加载完整数据（标签修正版） ==========
DATA_DIR = "rumor"
RUMOR_WEIBO_DIR = os.path.join(DATA_DIR, "rumor_weibo")
FACT_JSON_PATH = os.path.join(DATA_DIR, "fact.json")

# 收集谣言（负例）
rumor_list = []

# 微博谣言
for fpath in glob.glob(os.path.join(RUMOR_WEIBO_DIR, "*.json")):
    with open(fpath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        txt = data.get('rumorText')
        if txt and isinstance(txt, str):
            rumor_list.append(txt.strip())

# 收集事实（正例），用 set 去重
fact_set = set()

with open(FACT_JSON_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        explain = obj.get('explain', '')
        rumor = obj.get('rumor', '').strip()
        abstract = obj.get('abstract', '').strip()
        
        if explain in ['伪科学', '伪常识']:
            if rumor:
                rumor_list.append(rumor)   # 作为谣言
            if abstract:
                fact_set.add(abstract)     # 事实依据
        elif explain == '确实如此':
            if rumor:
                fact_set.add(rumor)        # 真实信息

# 事实列表
fact_list = list(fact_set)

print(f"谣言数: {len(rumor_list)}")
print(f"事实数: {len(fact_list)}")

all_texts = rumor_list + fact_list
all_labels = [1]*len(rumor_list) + [0]*len(fact_list)

# 去重
unique = {}
unique_texts, unique_labels = [], []
for t, l in zip(all_texts, all_labels):
    if t not in unique:
        unique[t] = True
        unique_texts.append(t)
        unique_labels.append(l)

print(f"去重后: 总{len(unique_texts)}, 谣言{sum(unique_labels)}, 事实{len(unique_labels)-sum(unique_labels)}")



完整数据集实验 (谣言386 vs 事实81)
谣言数: 335
事实数: 80
去重后: 总411, 谣言331, 事实80


In [4]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ========== 2. 数据预处理 ==========
print("\n 数据集划分...")
texts = unique_texts
labels = unique_labels

# 构建DataFrame以便处理
df = pd.DataFrame({'text': texts, 'label': labels})

# 严格去重，防止数据泄露
df = df.drop_duplicates(subset=['text'])

# 数据划分：训练集80%，验证集20%，随机种子固定以便复现
# 注意stratify=labels，确保训练集和验证集都具有相同的类别分布
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values
)

print(f"训练集样本数: {len(train_texts)}")
print(f"验证集样本数: {len(val_texts)}")
print(f"训练集类别分布: \n{pd.Series(train_labels).value_counts()}")
print(f"验证集类别分布: \n{pd.Series(val_labels).value_counts()}")


 数据集划分...
训练集样本数: 328
验证集样本数: 83
训练集类别分布: 
1    264
0     64
Name: count, dtype: int64
验证集类别分布: 
1    67
0    16
Name: count, dtype: int64


In [5]:
# ============================================================
# 步骤3：创建PyTorch Dataloader
# ============================================================
# 配置分词器
tokenizer = BertTokenizer.from_pretrained('./bert-base-chinese')
MAX_LEN = 128  # 根据你的文本长度设置，这里为128比较合适[reference:3]

class RumorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',  # 填充到最大长度
            truncation=True,       # 超过最大长度的截断
            return_tensors='pt'    # 返回PyTorch张量
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_data_loader(df_texts, df_labels, tokenizer, max_len, batch_size):
    dataset = RumorDataset(
        texts=df_texts,
        labels=df_labels,
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 这里的BATCH_SIZE取决于你的设备和显存，Colab上可以选择8或16[reference:4]
BATCH_SIZE = 8
train_data_loader = create_data_loader(train_texts, train_labels, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(val_texts, val_labels, tokenizer, MAX_LEN, BATCH_SIZE)


In [6]:
# ============================================================
# 步骤4：初始化BERT模型
# ============================================================
model = BertForSequenceClassification.from_pretrained('bert-base-chinese', num_labels=2) # 二分类任务[reference:5]
model = model.to(device)

# 优化器，使用AdamW[reference:6]
optimizer = AdamW(model.parameters(), lr=2e-5)

# 损失函数，二分类交叉熵[reference:7]
# loss_fn = torch.nn.CrossEntropyLoss().to(device)


# ============================================================
# 步骤5：训练模型
# ============================================================
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    return total_loss / len(data_loader)

def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return accuracy_score(actual_labels, predictions), predictions, actual_labels

# 微调持续3-4个epoch通常足以在小数据集上达到不错的效果，太长容易过拟合。
EPOCHS = 3
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 40)
    train_loss = train_epoch(model, train_data_loader, optimizer, device)
    print(f'Train loss: {train_loss:.4f}')
    acc, _, _ = eval_model(model, val_data_loader, device)
    print(f'Validation accuracy: {acc:.4f}')

print("训练完成！")


/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/transformers/optimization.py:457: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/3
----------------------------------------
Train loss: 0.3090
Validation accuracy: 0.9277
Epoch 2/3
----------------------------------------
Train loss: 0.1261
Validation accuracy: 0.9157
Epoch 3/3
----------------------------------------
Train loss: 0.0833
Validation accuracy: 0.9518
训练完成！


In [8]:
# ============================================================
# 步骤6：模型评估与结果对比
# ============================================================
print("\n评估BERT模型：")
_, predictions, actual = eval_model(model, val_data_loader, device)
print(classification_report(actual, predictions, target_names=['事实', '谣言']))
print("混淆矩阵：")
print(confusion_matrix(actual, predictions))

feature_best_acc = 0.9277  # 基于特征的最佳准确率
bert_acc = accuracy_score(actual, predictions)
print(f"\n特征方法的最佳准确率: {feature_best_acc:.3f}")
print(f"BERT方法的验证准确率: {bert_acc:.3f}")

if bert_acc > feature_best_acc:
    improvement = (bert_acc - feature_best_acc) / feature_best_acc
    print(f"✔️ BERT方法相比特征方法提升了 {(improvement*100):.1f}%")
else:
    print(f"✖️ 在小规模数据集上，简单方法可能已经足够好。")


评估BERT模型：
              precision    recall  f1-score   support

          事实       0.83      0.94      0.88        16
          谣言       0.98      0.96      0.97        67

    accuracy                           0.95        83
   macro avg       0.91      0.95      0.93        83
weighted avg       0.96      0.95      0.95        83

混淆矩阵：
[[15  1]
 [ 3 64]]

特征方法的最佳准确率: 0.928
BERT方法的验证准确率: 0.952
✔️ BERT方法相比特征方法提升了 2.6%
